# Part 3 - HMM POS Tagger with Viterbi

Corpus: NLTK Brown with the Universal tagset, 80% train / 20% test at sentence
level with the same seed as Part 2. Transitions and emissions are learned with
add-1 smoothing. Viterbi is implemented from scratch: no `nltk.tag.hmm`, no
`hmmlearn`. This notebook is self-contained.

Coverage: Q11 top-5 transitions and emissions, Q12 Viterbi from scratch,
Q13 seen vs unseen accuracy, Q14 ten mis-tagged tokens with error-type labels.

## Setup

In [1]:
from __future__ import annotations

import json
import logging
import math
import random
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [2]:
_REQUIRED_NLTK = (
    ("corpora/gutenberg", "gutenberg"),
    ("corpora/brown", "brown"),
    ("taggers/universal_tagset", "universal_tagset"),
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
)


def ensure_corpora() -> None:
    for resource_path, pkg in _REQUIRED_NLTK:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(pkg, quiet=True)


ensure_corpora()

In [3]:
@dataclass(frozen=True)
class Part3Config:
    corpus_name: str = "brown"
    tagset: str = "universal"
    seed: int = 42
    train_ratio: float = 0.8
    laplace_alpha: float = 1.0
    decode_log_every: int = 500
    error_examples: int = 10
    top_k: int = 5
    log_file: str = "part3_hmm.log"


cfg = Part3Config()
LOG_DIR = REPO / "logs"
ART_DIR = REPO / "artifacts" / "part3_hmm"
OUT_DIR = REPO / "outputs"
for d in (LOG_DIR, ART_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [4]:
_LOG_FORMAT = "%(asctime)s | %(levelname)-5s | %(name)s | %(message)s"
_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"


def get_logger(name: str, log_file: Path | str, level: str | int = "INFO") -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False
    for h in list(logger.handlers):
        logger.removeHandler(h)
        h.close()
    log_path = Path(log_file)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    formatter = logging.Formatter(_LOG_FORMAT, _DATE_FORMAT)
    fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    sh = logging.StreamHandler(stream=sys.stdout)
    sh.setFormatter(formatter)
    logger.addHandler(sh)
    return logger


def _format_metrics(metrics: Mapping[str, Any]) -> str:
    parts = []
    for k, v in metrics.items():
        parts.append(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}")
    return " ".join(parts)


def log_iteration(logger: logging.Logger, step: int, total: int | None,
                  prefix: str = "iter", **metrics: Any) -> None:
    header = f"{prefix} {step}/{total}" if total is not None else f"{prefix} {step}"
    body = _format_metrics(metrics)
    logger.info(f"{header} {body}".rstrip())


def log_section(logger: logging.Logger, title: str) -> None:
    logger.info(f"--- {title} ---")

In [5]:
def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, ensure_ascii=False)


def save_text(lines: Iterable[str], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for line in lines:
            fh.write(line.rstrip("\n") + "\n")

In [6]:
def split_sentences(sentences: Sequence, train_ratio: float, seed: int) -> Tuple[List, List]:
    rng = random.Random(seed)
    idx = list(range(len(sentences)))
    rng.shuffle(idx)
    cut = int(len(idx) * train_ratio)
    train = [sentences[i] for i in idx[:cut]]
    test = [sentences[i] for i in idx[cut:]]
    return train, test

## Corpus loader and HMM tagger

In [7]:
def load_tagged_sentences(corpus_name: str, tagset: str = "universal") -> List[List[Tuple[str, str]]]:
    if corpus_name != "brown":
        raise ValueError("tagged corpus only implemented for brown")
    return [list(s) for s in nltk.corpus.brown.tagged_sents(tagset=tagset)]

In [8]:
HMM_BOS, HMM_EOS = "<s>", "</s>"
NEG_INF = -1e18


class HMMTagger:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.tags: List[str] = []
        self._tag_idx: Dict[str, int] = {}
        self._trans_counts: Dict[str, Counter] = defaultdict(Counter)
        self._trans_totals: Dict[str, int] = defaultdict(int)
        self._emit_counts: Dict[str, Counter] = defaultdict(Counter)
        self._emit_totals: Dict[str, int] = defaultdict(int)
        self._train_vocab: set = set()
        self._V_emit = 0
        self._hapax_dist: Dict[str, float] = {}
        self._log_A: np.ndarray | None = None
        self._log_pi: np.ndarray | None = None
        self._log_final: np.ndarray | None = None

    def fit(self, train_sentences: Sequence[Sequence[Tuple[str, str]]],
            logger: logging.Logger | None = None) -> None:
        start = time.time()
        self._trans_counts.clear()
        self._trans_totals.clear()
        self._emit_counts.clear()
        self._emit_totals.clear()
        self._train_vocab.clear()
        word_counts: Counter = Counter()
        for sent in train_sentences:
            prev = HMM_BOS
            for word, tag in sent:
                self._trans_counts[prev][tag] += 1
                self._trans_totals[prev] += 1
                self._emit_counts[tag][word] += 1
                self._emit_totals[tag] += 1
                self._train_vocab.add(word)
                word_counts[word] += 1
                prev = tag
            self._trans_counts[prev][HMM_EOS] += 1
            self._trans_totals[prev] += 1
        self.tags = sorted(self._emit_counts.keys())
        self._tag_idx = {t: i for i, t in enumerate(self.tags)}
        self._V_emit = len(self._train_vocab)
        hapax = {w for w, c in word_counts.items() if c == 1}
        hapax_tag_counts = Counter()
        for tag, wc in self._emit_counts.items():
            for w, c in wc.items():
                if w in hapax:
                    hapax_tag_counts[tag] += c
        total_hapax = sum(hapax_tag_counts.values())
        if total_hapax == 0:
            self._hapax_dist = {t: 1.0 / len(self.tags) for t in self.tags}
        else:
            self._hapax_dist = {
                t: (hapax_tag_counts.get(t, 0) + self.alpha)
                / (total_hapax + self.alpha * len(self.tags))
                for t in self.tags
            }
        self._precompute_log_matrices()
        if logger is not None:
            log_iteration(logger, 1, 1, prefix="fit",
                          sentences=len(train_sentences),
                          tags=len(self.tags), vocab=self._V_emit,
                          trans_contexts=len(self._trans_counts),
                          elapsed_s=time.time() - start)

    def _precompute_log_matrices(self) -> None:
        T = len(self.tags)
        alpha = self.alpha
        log_A = np.full((T, T), NEG_INF, dtype=np.float64)
        for i, prev in enumerate(self.tags):
            denom = self._trans_totals.get(prev, 0) + alpha * (T + 1)
            for j, nxt in enumerate(self.tags):
                num = self._trans_counts[prev].get(nxt, 0) + alpha
                log_A[i, j] = math.log(num) - math.log(denom)
        self._log_A = log_A
        log_pi = np.full(T, NEG_INF, dtype=np.float64)
        denom_bos = self._trans_totals.get(HMM_BOS, 0) + alpha * (T + 1)
        for j, tag in enumerate(self.tags):
            num = self._trans_counts[HMM_BOS].get(tag, 0) + alpha
            log_pi[j] = math.log(num) - math.log(denom_bos)
        self._log_pi = log_pi
        log_final = np.full(T, NEG_INF, dtype=np.float64)
        for i, tag in enumerate(self.tags):
            denom = self._trans_totals.get(tag, 0) + alpha * (T + 1)
            num = self._trans_counts[tag].get(HMM_EOS, 0) + alpha
            log_final[i] = math.log(num) - math.log(denom)
        self._log_final = log_final

    def _log_emit_column(self, word: str) -> np.ndarray:
        T = len(self.tags)
        col = np.empty(T, dtype=np.float64)
        alpha = self.alpha
        if word in self._train_vocab:
            for j, tag in enumerate(self.tags):
                num = self._emit_counts[tag].get(word, 0) + alpha
                den = self._emit_totals.get(tag, 0) + alpha * self._V_emit
                col[j] = math.log(num) - math.log(den)
        else:
            for j, tag in enumerate(self.tags):
                p = self._hapax_dist.get(tag, 1e-12)
                col[j] = math.log(max(p, 1e-12))
        return col

    def top_transitions(self, from_tag: str, k: int) -> List[Tuple[str, float]]:
        totals = self._trans_totals.get(from_tag, 0)
        denom = totals + self.alpha * (len(self.tags) + 1)
        probs = []
        for tag in list(self.tags) + [HMM_EOS]:
            num = self._trans_counts[from_tag].get(tag, 0) + self.alpha
            probs.append((tag, num / denom))
        probs.sort(key=lambda x: x[1], reverse=True)
        return probs[:k]

    def top_emissions(self, from_tag: str, k: int) -> List[Tuple[str, float]]:
        total = self._emit_totals.get(from_tag, 0)
        denom = total + self.alpha * self._V_emit
        probs = [(w, (c + self.alpha) / denom)
                 for w, c in self._emit_counts[from_tag].items()]
        probs.sort(key=lambda x: x[1], reverse=True)
        return probs[:k]

    def decode(self, sentence: Sequence[str]) -> List[str]:
        if not sentence:
            return []
        assert self._log_A is not None and self._log_pi is not None and self._log_final is not None
        T = len(self.tags)
        n = len(sentence)
        score = np.full((n, T), NEG_INF, dtype=np.float64)
        back = np.full((n, T), -1, dtype=np.int32)
        score[0] = self._log_pi + self._log_emit_column(sentence[0])
        for t in range(1, n):
            e_t = self._log_emit_column(sentence[t])
            trans_scores = score[t - 1][:, None] + self._log_A
            best_prev = np.argmax(trans_scores, axis=0)
            best_val = trans_scores[best_prev, np.arange(T)]
            score[t] = best_val + e_t
            back[t] = best_prev
        final_scores = score[n - 1] + self._log_final
        best_last = int(np.argmax(final_scores))
        path_idx = [best_last]
        for t in range(n - 1, 0, -1):
            path_idx.append(int(back[t, path_idx[-1]]))
        path_idx.reverse()
        return [self.tags[i] for i in path_idx]

    def decode_many(self, sentences: Sequence[Sequence[str]],
                    logger: logging.Logger | None = None,
                    log_every: int = 500) -> List[List[str]]:
        out: List[List[str]] = []
        start = time.time()
        for i, s in enumerate(sentences, start=1):
            out.append(self.decode(s))
            if logger is not None and (i % log_every == 0 or i == len(sentences)):
                log_iteration(logger, i, len(sentences), prefix="decode",
                              elapsed_s=time.time() - start)
        return out


def evaluate_predictions(gold_sentences: Sequence[Sequence[Tuple[str, str]]],
                         pred_sentences: Sequence[Sequence[str]],
                         train_vocab: set) -> Dict[str, float | int]:
    total = seen_total = unseen_total = 0
    correct = seen_correct = unseen_correct = 0
    for gold, pred in zip(gold_sentences, pred_sentences):
        for (w, g), p in zip(gold, pred):
            total += 1
            hit = int(g == p)
            correct += hit
            if w in train_vocab:
                seen_total += 1
                seen_correct += hit
            else:
                unseen_total += 1
                unseen_correct += hit
    return {
        "tokens": total, "correct": correct,
        "accuracy": correct / max(1, total),
        "seen_tokens": seen_total, "seen_correct": seen_correct,
        "seen_accuracy": seen_correct / max(1, seen_total),
        "unseen_tokens": unseen_total, "unseen_correct": unseen_correct,
        "unseen_accuracy": unseen_correct / max(1, unseen_total),
    }


def _classify_error(word: str, gold: str, pred: str, train_vocab: set, tagger: HMMTagger) -> str:
    if word not in train_vocab:
        return "OOV"
    emit_for_word = {t: tagger._emit_counts[t].get(word, 0) for t in tagger.tags}
    seen_tags = [t for t, c in emit_for_word.items() if c > 0]
    if len(seen_tags) > 1:
        return "lexical ambiguity"
    if pred not in seen_tags:
        return "sparse transition"
    return "tag frequency bias"


def collect_errors(gold_sentences: Sequence[Sequence[Tuple[str, str]]],
                   pred_sentences: Sequence[Sequence[str]],
                   train_vocab: set, tagger: HMMTagger, limit: int) -> List[Dict[str, object]]:
    errors: List[Dict[str, object]] = []
    for gold, pred in zip(gold_sentences, pred_sentences):
        for (w, g), p in zip(gold, pred):
            if g == p or len(errors) >= limit:
                continue
            errors.append({
                "token": w, "gold": g, "pred": p,
                "seen_in_train": w in train_vocab,
                "error_type": _classify_error(w, g, p, train_vocab, tagger),
            })
        if len(errors) >= limit:
            break
    return errors

## Load and split

In [9]:
logger = get_logger("part3_hmm", LOG_DIR / cfg.log_file)
logger.info(f"config={cfg}")

log_section(logger, "load and split tagged corpus")
tagged = load_tagged_sentences(cfg.corpus_name, tagset=cfg.tagset)
train, test = split_sentences(tagged, train_ratio=cfg.train_ratio, seed=cfg.seed)
logger.info(f"sentences total={len(tagged):,} train={len(train):,} test={len(test):,}")
logger.info(f"train_tokens={sum(len(s) for s in train):,} test_tokens={sum(len(s) for s in test):,}")

2026-07-05 14:20:06 | INFO  | part3_hmm | config=Part3Config(corpus_name='brown', tagset='universal', seed=42, train_ratio=0.8, laplace_alpha=1.0, decode_log_every=500, error_examples=10, top_k=5, log_file='part3_hmm.log')


2026-07-05 14:20:06 | INFO  | part3_hmm | --- load and split tagged corpus ---


2026-07-05 14:20:07 | INFO  | part3_hmm | sentences total=57,340 train=45,872 test=11,468


2026-07-05 14:20:07 | INFO  | part3_hmm | train_tokens=929,015 test_tokens=232,177


## Q11 - Learned parameters

In [10]:
log_section(logger, "fit HMM")
tagger = HMMTagger(alpha=cfg.laplace_alpha)
tagger.fit(train, logger=logger)

top_trans = tagger.top_transitions("NOUN", k=cfg.top_k)
top_emit = tagger.top_emissions("VERB", k=cfg.top_k)

trans_df = pd.DataFrame(top_trans, columns=["next_tag", "prob"])
trans_df.insert(0, "prev_tag", "NOUN")
emit_df = pd.DataFrame(top_emit, columns=["word", "prob"])
emit_df.insert(0, "tag", "VERB")

save_dataframe(trans_df, ART_DIR / "top_transitions_from_NOUN.csv")
save_dataframe(emit_df, ART_DIR / "top_emissions_from_VERB.csv")

logger.info(f"top NOUN->: {top_trans}")
logger.info(f"top VERB emissions: {top_emit}")

print("Top-5 transitions from NOUN:")
print(trans_df.to_string(index=False))
print()
print("Top-5 emissions from VERB:")
print(emit_df.to_string(index=False))

2026-07-05 14:20:07 | INFO  | part3_hmm | --- fit HMM ---


2026-07-05 14:20:08 | INFO  | part3_hmm | fit 1/1 sentences=45872 tags=12 vocab=50630 trans_contexts=13 elapsed_s=0.4790


2026-07-05 14:20:08 | INFO  | part3_hmm | top NOUN->: [('.', 0.2836201228774173), ('ADP', 0.24387766668933777), ('VERB', 0.15869323720782608), ('NOUN', 0.14994672289101998), ('CONJ', 0.0596567593915074)]


2026-07-05 14:20:08 | INFO  | part3_hmm | top VERB emissions: [('is', 0.04084320135855887), ('was', 0.04006528404150926), ('be', 0.025981421504075168), ('had', 0.020683448665083714), ('are', 0.017388740028167725)]


Top-5 transitions from NOUN:
prev_tag next_tag     prob
    NOUN        . 0.283620
    NOUN      ADP 0.243878
    NOUN     VERB 0.158693
    NOUN     NOUN 0.149947
    NOUN     CONJ 0.059657

Top-5 emissions from VERB:
 tag word     prob
VERB   is 0.040843
VERB  was 0.040065
VERB   be 0.025981
VERB  had 0.020683
VERB  are 0.017389


## Q12 - Viterbi decode on the test set

`HMMTagger.decode` implements Viterbi from scratch: it maintains a score
matrix and a back-pointer matrix, operates in log space to avoid underflow,
and backtracks from the best terminal cell. `decode_many` streams progress to
the log every `decode_log_every` sentences.

In [11]:
log_section(logger, "viterbi decode on test set")
test_words = [[w for (w, _) in s] for s in test]
pred_tags = tagger.decode_many(test_words, logger=logger, log_every=cfg.decode_log_every)

2026-07-05 14:20:08 | INFO  | part3_hmm | --- viterbi decode on test set ---


2026-07-05 14:20:08 | INFO  | part3_hmm | decode 500/11468 elapsed_s=0.0800


2026-07-05 14:20:08 | INFO  | part3_hmm | decode 1000/11468 elapsed_s=0.1530


2026-07-05 14:20:08 | INFO  | part3_hmm | decode 1500/11468 elapsed_s=0.2220


2026-07-05 14:20:08 | INFO  | part3_hmm | decode 2000/11468 elapsed_s=0.2950


...

2026-07-05 14:20:09 | INFO  | part3_hmm | decode 9500/11468 elapsed_s=1.4860


2026-07-05 14:20:09 | INFO  | part3_hmm | decode 10000/11468 elapsed_s=1.5560


2026-07-05 14:20:10 | INFO  | part3_hmm | decode 10500/11468 elapsed_s=1.6280


2026-07-05 14:20:10 | INFO  | part3_hmm | decode 11000/11468 elapsed_s=1.7033


2026-07-05 14:20:10 | INFO  | part3_hmm | decode 11468/11468 elapsed_s=1.7703


## Q13 - Token-level accuracy (overall / seen / unseen)

In [12]:
log_section(logger, "evaluate")
metrics = evaluate_predictions(test, pred_tags, tagger._train_vocab)
save_json(metrics, ART_DIR / "accuracy_summary.json")
logger.info(f"accuracy overall={metrics['accuracy']:.4f} seen={metrics['seen_accuracy']:.4f} unseen={metrics['unseen_accuracy']:.4f}")

acc_df = pd.DataFrame([
    {"bucket": "overall", "tokens": metrics["tokens"], "correct": metrics["correct"], "accuracy": metrics["accuracy"]},
    {"bucket": "seen", "tokens": metrics["seen_tokens"], "correct": metrics["seen_correct"], "accuracy": metrics["seen_accuracy"]},
    {"bucket": "unseen", "tokens": metrics["unseen_tokens"], "correct": metrics["unseen_correct"], "accuracy": metrics["unseen_accuracy"]},
])
save_dataframe(acc_df, ART_DIR / "accuracy_table.csv")
acc_df

2026-07-05 14:20:10 | INFO  | part3_hmm | --- evaluate ---


2026-07-05 14:20:10 | INFO  | part3_hmm | accuracy overall=0.9402 seen=0.9482 unseen=0.6308


,bucket,tokens,correct,accuracy
0,overall,232177,218294,0.940205
1,seen,226311,214594,0.948226
2,unseen,5866,3700,0.630753


## Q14 - Error analysis (10 mis-tagged tokens)

In [13]:
log_section(logger, "error analysis")
errors = collect_errors(test, pred_tags, tagger._train_vocab, tagger, limit=cfg.error_examples)
err_df = pd.DataFrame(errors)
save_dataframe(err_df, ART_DIR / "error_analysis.csv")
type_counts = err_df["error_type"].value_counts().to_dict()
save_json({"errors": errors, "type_counts": type_counts}, ART_DIR / "error_analysis.json")
logger.info(f"error_type_counts={type_counts}")
err_df

2026-07-05 14:20:10 | INFO  | part3_hmm | --- error analysis ---


2026-07-05 14:20:10 | INFO  | part3_hmm | error_type_counts={'lexical ambiguity': 5, 'sparse transition': 3, 'OOV': 2}


,token,gold,pred,seen_in_train,error_type
0,average,VERB,NOUN,True,lexical ambiguity
1,$200,NOUN,ADP,True,sparse transition
2,depending,ADP,VERB,True,lexical ambiguity
3,very,ADV,ADJ,True,lexical ambiguity
4,tempting,VERB,NOUN,True,sparse transition
5,offer,NOUN,VERB,True,lexical ambiguity
6,as,ADP,ADV,True,lexical ambiguity
7,Add,VERB,DET,True,sparse transition
8,non-partisan,ADJ,NOUN,False,OOV
9,cosy,ADJ,NOUN,False,OOV


## Persist to outputs/

In [14]:
for src_name, dst_name in [
    ("top_transitions_from_NOUN.csv", "part3_transitions_from_NOUN.csv"),
    ("top_emissions_from_VERB.csv", "part3_emissions_from_VERB.csv"),
    ("accuracy_summary.json", "part3_accuracy.json"),
    ("accuracy_table.csv", "part3_accuracy.csv"),
    ("error_analysis.csv", "part3_errors.csv"),
]:
    (OUT_DIR / dst_name).write_bytes((ART_DIR / src_name).read_bytes())

logger.info("part 3 done - artifacts written to artifacts/part3_hmm and outputs/")

2026-07-05 14:20:10 | INFO  | part3_hmm | part 3 done - artifacts written to artifacts/part3_hmm and outputs/
